# **Use of Differential Equations to calculate the population change over time in Disease Modeling**

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

In [ ]:
# Part A: ODE Disease-Transmission Models
# ---------------------------------------------------------------
# 1. Classic SIR model
#    dS/dt = -beta*S*I/N
#    dI/dt =  beta*S*I/N - gamma*I
#    dR/dt =  gamma*I
# ---------------------------------------------------------------

def sir(state, t, N, beta, gamma):
    S, I, R = state
    dSdt = -beta * S * I / N  # Change in S population over time
    dIdt = beta * S * I / N - gamma * I  # Change in I population over time
    dRdt = gamma * I # Change in R population over time
    return dSdt, dIdt, dRdt

Suppose we have a population of N=1000 people, of which one is infectious with some disease, and none have immunity.

Effective contact rate is transmission rate * contact rate, so:

For example, 5% transmission rate and 5 contacts a day is 0.05 * 5 = 0.25,

Recovery rate is 1 / days, so for example, 4 day recovery rate is 1 / 4 = 0.25

In [ ]:
# Paramater Beta: Effective contact rate
# Parameter Gamma : Recovery rate

beta = 0.15 # Set it differently to see the changes
gamma = 1/21

# Calculate Reproduction Number (R0)
print("R0 is", beta / gamma)

# What is our start population look like?
# Everyone not infected or recovered is susceptible
# Supply of starting values for each compartment in SIR model
N = 100000
R = 0
I = 26
S = N - I - R

# A list of days (0 to 160 days)
days = range(0, 160)


# Applying differential equations with our population
sir_mod = odeint(sir,
             [S, I, R],
             days,
             args=(N, beta, gamma)) # Here 'args' is a tuple that passing the information and other arguments to the deriv function
S, I, R = sir_mod.T # as we have created sir_mod variable, putting .T acting as transpose where rows and collums are getting swapped.

# Build a dataframe and label state variables
sir_df = pd.DataFrame({
    'Susceptible (S)': S,
    'Infected (I)': I,
    'Recovered (R)': R,
    'Day': days
})

# Calculate the maximum number of people sick at one time
max_infected = sir_df['Infected (I)'].max()
print(f"The maximum number of people sick at one time is approximately {max_infected:.0f}.")


# Plot the SIR
plt.style.use('ggplot')
sir_plot = sir_df.plot(x='Day',
        y=['Susceptible (S)', 'Infected (I)', 'Recovered (R)'],
        color=['Yellow', 'Red', 'Green'], # set your own colour
        kind='line', # Change 'line' into 'area' for area coverage
        stacked=False) # Changed stacked=True to stacked=False
plt.title(f'SIR model, (R0={beta/gamma:.1f})')
plt.xlabel('Days')
plt.ylabel('Population')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# If you get the error:
#
#     When stacked is True, each column must be either all
#     positive or negative.infected contains both...
#
# just change stacked=True to stacked=False

In [ ]:
# ---------------------------------------------------------------
# 2. SEIR model (adds Exposed / incubation compartment)
#    dS/dt = -beta*S*I/N
#    dE/dt =  beta*S*I/N - sigma*E
#    dI/dt =  sigma*E - gamma*I
#    dR/dt =  gamma*I
# ---------------------------------------------------------------
sigma = 1/5.2   # incubation rate (~5.2 day incubation, COVID-like)
def seir(t, state, beta, sigma, gamma, N):
    S,E,I,R = state
    dS = -beta*S*I/N
    dE = beta*S*I/N - sigma*E
    dI = sigma*E - gamma*I
    dR = gamma*I
    return [dS,dE,dI,dR]

In [ ]:
# Initial conditions for SEIR model
# Using N, beta, gamma from SIR model and sigma from SEIR definition

I0 = 26  # Initial infected individuals
E0 = 0   # Initial exposed individuals
R0 = 0   # Initial recovered individuals
S0 = N - I0 - E0 - R0 # Initial susceptible individuals

y0 = [S0, E0, I0, R0] # Initial state vector [S, E, I, R]

# Time span for the simulation
t_span = [0, 200] # from 0 to 200 days
t_eval = np.linspace(t_span[0], t_span[1], 400) # Evaluate at 400 points

# Solve the SEIR differential equations
seir_mod = solve_ivp(
    seir, t_span, y0,
    args=(beta, sigma, gamma, N),
    dense_output=True,
    t_eval=t_eval
    )

# Extract the results
S_seir, E_seir, I_seir, R_seir = seir_mod.y

# Build a dataframe and label state variables (corrected to use time-series data)
seir_df = pd.DataFrame({
    'Susceptible (S)': S_seir,
    'Exposed (E)': E_seir,
    'Infected (I)': I_seir,
    'Recovered (R)': R_seir,
    'Days': seir_mod.t
})

In [ ]:
# Plot the SEIR
plt.style.use('ggplot')
seir_plot = seir_df.plot(x='Days',
        y=['Susceptible (S)','Exposed (E)','Infected (I)','Recovered (R)'],
        color=['yellow', 'orange', 'red', 'green'], # set your own colour
        kind='line', # Change 'line' into 'area' for area coverage
        stacked=False) # Changed stacked=True to stacked=False

for i, l in enumerate(labels):
    plt.plot(seir_mod.t, seir_mod.y[i], label=l, color=colors[i])

plt.title(f'SEIR model (incubation ~{1/sigma:.1f} days, R0={beta/gamma:.1f})')
plt.xlabel('Days')
plt.ylabel('Population')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()